In [ ]:
!pip uninstall -y langchain langchain-core langchain-community langchain-classic langgraph langgraph-sdk langgraph-prebuilt groq langchain-groq httpx chromadb opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc opentelemetry-instrumentation
!pip install -q langchain==0.2.17 langchain-core==0.2.43 langchain-community==0.2.19 langchain-groq==0.1.9 groq==0.9.0 langchain-google-genai groq==0.9.0 langchain-groq==0.1.9 httpx==0.27.0 langchain-huggingface==0.0.3 chromadb==0.5.23 opentelemetry-api==1.27.0 opentelemetry-sdk==1.27.0 gdown rich


Found existing installation: langchain 1.3.6
Uninstalling langchain-1.3.6:
  Successfully uninstalled langchain-1.3.6
Found existing installation: langchain-core 1.4.3
Uninstalling langchain-core-1.4.3:
  Successfully uninstalled langchain-core-1.4.3
Found existing installation: langgraph 1.2.4
Uninstalling langgraph-1.2.4:
  Successfully uninstalled langgraph-1.2.4
Found existing installation: langgraph-sdk 0.4.2
Uninstalling langgraph-sdk-0.4.2:
  Successfully uninstalled langgraph-sdk-0.4.2
Found existing installation: langgraph-prebuilt 1.1.0
Uninstalling langgraph-prebuilt-1.1.0:
  Successfully uninstalled langgraph-prebuilt-1.1.0
Found existing installation: httpx 0.28.1
Uninstalling httpx-0.28.1:
  Successfully uninstalled httpx-0.28.1
Found existing installation: opentelemetry-api 1.38.0
Uninstalling opentelemetry-api-1.38.0:
  Successfully uninstalled opentelemetry-api-1.38.0
Found existing installation: opentelemetry-sdk 1.38.0
Uninstalling opentelemetry-sdk-1.38.0:
  Success

In [15]:
import os
from typing import List
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from getpass import getpass
from langchain_groq import ChatGroq
groq_key = getpass("Enter Groq API Key: ")

llm = ChatGroq(
    api_key=groq_key,
    model="llama-3.1-8b-instant",
    temperature=0.2
)



Enter Groq API Key: ··········


In [16]:
load_dotenv()

#1. Data Models

class MCQ(BaseModel):
  question: str = Field(description="Multiple-choice question")
  options: List[str] = Field(description="Four answer options")
  correct_answer: str = Field(description="Correct option text")
  explanation: str = Field(description="Explanation of the correct answer")

class QuizOutput(BaseModel):
  Quiz: List[MCQ] = Field(description="List of generated quiz questions")

#2. Load PDF
def load_pdf_text(pdf_path: str) -> str:
  loader=PyPDFLoader(pdf_path)
  pages=loader.load()
  text = "\n\n".join(page.page_content for page in pages)
  return clean_text(text)

def clean_text(text: str) -> str:
  text=text.replace("\x100"," ")
  text=" ".join(text.split())
  return text


#3. Chunk Text
def chunk_text(text: str, max_chars: int = 5000) -> List[str]:
  chunks = []
  start=0

  while start < len(text):
    end = start + max_chars
    chunks.append(text[start:end])
    start = end

  return chunks

#4. Summarization Chain

def summarize_material(text: str) -> str:
  summary_prompt=ChatPromptTemplate.from_messages([
      (
          "system",
          "You are an expert study assistant.Summarize study material into clear, concise, exam-friendly bullet points."

      ),
      ( "human", """
      Summarize the following study material.

      Requirments:
      - Keep only important concepts.
      - Use simple language.
      - Preserve definitions, techniques examples and warnings.
      - Avoid unnecessary repetition.

      Study Material:
      {content}""" )
  ])


  chain = summary_prompt | llm
  chunks=chunk_text(text)
  partial_summaries = []
  for chunk in chunks:
    response = chain.invoke({"content": chunk})
    partial_summaries.append(response.content)

  combined_summary = "\n\n".join(partial_summaries)

  final_prompt=ChatPromptTemplate.from_messages([
      ("system",
       "You are an expert educator.Combine partial summaries into one clean final study summary."),
      ("human",
       """
       Combine and refine these summaries into one final concise summary.

       Requirments:
       - Remove duplicates.
       - Organize by topic.
       - Keep bullet points.
       - Make it useful for revision.

       Partial Summaries:
       {summaries}
       """)


  ])

  final_chain = final_prompt | llm
  final_response = final_chain.invoke({"summaries": combined_summary})
  return final_response.content


In [23]:
# 5.Quiz Generation Chain
def generate_quiz(summary: str, num_questions: int=5) -> QuizOutput:
  structured_llm=llm.with_structured_output(QuizOutput)
  quiz_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are an expert quiz generator.
        Create high quality multiple choice questions based on the following study material.
        Each question must test understanding, not just memorization.
        """
    ),
    (
        "human",
        """
        Generate {num_questions} multiple choice questions from the study summary below.

        Rules:
        - Each question must have exactly 4 options.
        - Only one option should be correct.
        - Options should be plausible.
        - Include the correct answer.
        - Include a short explanation.
        - Avoid vague or trick questions.

        Study Summary:
        {summary}
        """
    )
])

  chain=quiz_prompt | structured_llm
  quiz_output=chain.invoke({"num_questions": num_questions, "summary": summary})
  return quiz_output

  #6. Main Exceution

if __name__=="__main__":
  pdf_path="Prompt Engineering.pdf"
  study_text=load_pdf_text(pdf_path)

  print("\n Generating Summary...\n")
  summary=summarize_material(study_text)
  print("SUMMARY")
  print("=" * 80)
  print(summary)


  print("\n Generating Quiz...\n")
  quiz=generate_quiz(summary, num_questions=5)

  print("QUIZ")
  print("=" * 80)

  for i, item in enumerate(quiz.Quiz, start=1):
    print(f"Question {i}: {item.question}")
    for idx, option in enumerate(item.options, start=1):
      print(f"{idx}. {option}")
    print(f"Correct Answer: {item.correct_answer}")
    print(f"Explanation: {item.explanation}")
    print("=" * 80)




 Generating Summary...

SUMMARY
**Final Study Summary: Prompt Engineering in AI**

**What is Prompt Engineering?**

* A practical application of Natural Language Processing (NLP) in Artificial Intelligence (AI)
* Text is used to describe the task the AI should perform
* Goal: to use human-understandable text to interact conversationally with models, allowing for flexibility in the model's performance due to the task description embedded in the prompt

**Key Elements of a Prompt:**

* **Instruction:** A statement telling the model what task to perform
* **Context:** Background information that helps the model understand the problem at hand
* **Input Data:** The input data given to the model to process
* **Output Indicator:** A specification of the expected output format (e.g., code, text, image)

**Standard Prompt Patterns:**

* **User-Model Interaction:** User: <Instruction> Model: <Response>
* **Few-Shot Prompting:** Provides a few examples of the task to guide the model
* **Question

In [19]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 7.1 MB/s eta 0:00:00


In [20]:
pdf_path="Prompt Engineering.pdf"
study_text=load_pdf_text(pdf_path)

print("\n Generating Summary...\n")
print("Total characters:", len(study_text))
summary=summarize_material(study_text)
print("SUMMARY")
print("=" * 80)
print(summary)


 Generating Summary...

Total characters: 4302
SUMMARY
**Prompt Engineering: Key Concepts and Techniques**

**What is Prompt Engineering?**

* A practice within Natural Language Processing (NLP) in Artificial Intelligence (AI) that uses human-understandable text to interact conversationally with models.

**Key Concepts:**

* **Prompts**: Detailed descriptions of the desired output from an AI model, guiding the model's performance.
* **Elements of a Prompt**: 
  * **Instruction**: A statement telling the model what task to perform.
  * **Context**: Background information that helps the model understand the problem at hand.
  * **Input Data**: The input data given to the model to process.
  * **Output Indicator**: A specification of the expected output format (e.g., code, text, image).

**Designing Effective Prompts:**

* **Role Playing**: Make the model act as a specific entity (e.g., teacher, code editor, interviewer) to tailor the interaction and target a specific outcome.
* **Clarit

In [24]:
quiz=generate_quiz(summary, num_questions=5)

print("QUIZ")
print("=" * 80)

for i, item in enumerate(quiz.Quiz, start=1):
  print(f"Question {i}: {item.question}")
  for idx, option in enumerate(item.options, start=1):
    print(f"{idx}. {option}")
  print(f"Correct Answer: {item.correct_answer}")
  print(f"Explanation: {item.explanation}")

QUIZ
Question 1: What is the primary goal of Prompt Engineering in AI?
1. To improve the performance of AI models through better data
2. To use human-understandable text to interact conversationally with models
3. To reduce the complexity of AI algorithms
4. To increase the processing power of AI models
Correct Answer: To use human-understandable text to interact conversationally with models
Explanation: The primary goal of Prompt Engineering is to use human-understandable text to interact conversationally with AI models, allowing for flexibility in the model's performance due to the task description embedded in the prompt.
Question 2: What is the purpose of the Instruction element in a prompt?
1. To provide background information
2. To specify the expected output format
3. To tell the model what task to perform
4. To give the model input data
Correct Answer: To tell the model what task to perform
Explanation: The Instruction element in a prompt is used to tell the model what task to p